# Ablation — CNN With vs Without Noise Augmentation
Same 70/15/15 split as Week 4 (seed 42), same `Conv1DClassifier` architecture. The only variable changed is: training-time Gaussian noise added to the flux arrays, on vs off. Val/test sets are never augmented — augmentation only changes what the model sees during training, not what it's evaluated against.

**Noise design:** per-sample adaptive Gaussian noise, `noise_std = ALPHA * sample_flux.std()`, re-sampled fresh every epoch (not a fixed static copy) — exposes the model to more variation each pass rather than just memorizing one noisy version. A fixed absolute noise value was avoided because transit depths vary a lot across TCEs; a fixed value could be negligible for some samples and swamp the real signal in others.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, re, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, confusion_matrix)
from tqdm import tqdm

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

DRIVE_ROOT = '/content/drive/MyDrive/SADA'
CSV_PATH   = os.path.join(DRIVE_ROOT, 'balanced_tce.csv')
NPY_DIR    = os.path.join(DRIVE_ROOT, 'processed')
RESULTS_PATH = os.path.join(DRIVE_ROOT, 'model_results.csv')

## 1. Rebuild the exact same row set + split as Week 4 (identical test set)

In [ ]:
balanced = pd.read_csv(CSV_PATH)

npy_files = [f for f in os.listdir(NPY_DIR) if f.endswith('.npy') and f != 'failed_kepids.npy']
pattern = re.compile(r'^(\d+)_(\d+)_(PC|FP)\.npy$')
records = []
for fname in npy_files:
    m = pattern.match(fname)
    if not m:
        continue
    kepid, plnt_num, label_str = m.groups()
    records.append({'kepid': int(kepid), 'tce_plnt_num': int(plnt_num),
                     'label': 1 if label_str == 'PC' else 0,
                     'npy_path': os.path.join(NPY_DIR, fname)})
npy_index = pd.DataFrame(records)

data = pd.merge(npy_index, balanced, on=['kepid', 'tce_plnt_num'], how='inner')
data = data.dropna(subset=['tce_period','tce_duration','tce_depth','tce_model_snr']).reset_index(drop=True)
print(f"Row set: {len(data)}")

y = data['label'].values
idx_train, idx_temp = train_test_split(data.index, test_size=0.30, random_state=SEED, stratify=y)
idx_val, idx_test = train_test_split(idx_temp, test_size=0.50, random_state=SEED, stratify=y[idx_temp])
print(f"Train: {len(idx_train)} | Val: {len(idx_val)} | Test: {len(idx_test)}")

## 2. Preload flux arrays into memory once (same fix as Week 4's Drive I/O issue)

In [ ]:
flux_matrix = []
for path in tqdm(data['npy_path'], desc='Preloading flux arrays'):
    flux_matrix.append(np.load(path).astype(np.float32))
flux_matrix = np.stack(flux_matrix)
labels_all = data['label'].values.astype(np.float32)
print(f"Preloaded shape: {flux_matrix.shape}")

## 3. Dataset with optional per-sample adaptive noise augmentation
`augment=True` only ever used for the training split. Noise is re-drawn fresh on every `__getitem__` call (i.e. every epoch sees a different noisy version), which is what makes this an augmentation rather than just adding static noise to the dataset once.

In [ ]:
ALPHA = 0.3   # noise_std = ALPHA * this sample's own flux std

class FluxDataset(Dataset):
    def __init__(self, indices, augment=False, alpha=ALPHA):
        self.indices = indices
        self.augment = augment
        self.alpha = alpha

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        idx = self.indices[i]
        flux = flux_matrix[idx].copy()
        if self.augment:
            noise_std = self.alpha * flux.std()
            flux = flux + np.random.normal(0, noise_std, size=flux.shape).astype(np.float32)
        flux_t = torch.tensor(flux).unsqueeze(0)
        label_t = torch.tensor(labels_all[idx])
        return flux_t, label_t

# quick sanity check: how much does the noise actually perturb a sample?
sample = flux_matrix[idx_train[0]]
print(f"Example sample std: {sample.std():.6f} | noise_std added: {ALPHA * sample.std():.6f}")

## 4. Model + training function (identical architecture/hyperparameters to Week 4)

In [ ]:
class Conv1DClassifier(nn.Module):
    def __init__(self, dropout=0.3):
        super().__init__()
        self.block1 = nn.Sequential(nn.Conv1d(1, 8, 5, padding=2), nn.BatchNorm1d(8),
                                     nn.ReLU(), nn.MaxPool1d(2), nn.Dropout(dropout))
        self.block2 = nn.Sequential(nn.Conv1d(8, 16, 5, padding=2), nn.BatchNorm1d(16),
                                     nn.ReLU(), nn.MaxPool1d(2), nn.Dropout(dropout))
        self.block3 = nn.Sequential(nn.Conv1d(16, 32, 5, padding=2), nn.BatchNorm1d(32),
                                     nn.ReLU(), nn.MaxPool1d(2), nn.Dropout(dropout))
        self.gap = nn.AdaptiveAvgPool1d(1)
        self.head = nn.Sequential(nn.Linear(32, 16), nn.ReLU(),
                                   nn.Dropout(dropout), nn.Linear(16, 1))

    def forward(self, x):
        x = self.block1(x); x = self.block2(x); x = self.block3(x)
        x = self.gap(x).squeeze(-1)
        return self.head(x).squeeze(-1)

def train_and_evaluate(name, augment_train, max_epochs=60, patience=8):
    torch.manual_seed(SEED)  # reset seed so both runs start from the same init
    model = Conv1DClassifier().to(device)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

    train_loader = DataLoader(FluxDataset(idx_train, augment=augment_train), batch_size=64, shuffle=True)
    val_loader   = DataLoader(FluxDataset(idx_val, augment=False), batch_size=64, shuffle=False)
    test_loader  = DataLoader(FluxDataset(idx_test, augment=False), batch_size=64, shuffle=False)

    best_val_loss, best_state, no_improve = float('inf'), None, 0
    history = {'train_loss': [], 'val_loss': []}

    for epoch in range(max_epochs):
        model.train()
        train_losses = []
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward(); optimizer.step()
            train_losses.append(loss.item())

        model.eval()
        val_losses = []
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device)
                val_losses.append(criterion(model(xb), yb).item())

        train_loss, val_loss = np.mean(train_losses), np.mean(val_losses)
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f"  [{name}] early stopping at epoch {epoch+1}")
                break

    model.load_state_dict(best_state)

    # evaluate on the untouched, non-augmented test set
    model.eval()
    probs, labels = [], []
    with torch.no_grad():
        for xb, yb in test_loader:
            xb = xb.to(device)
            probs.extend(torch.sigmoid(model(xb)).cpu().numpy())
            labels.extend(yb.numpy())
    probs, labels = np.array(probs), np.array(labels)
    preds = (probs > 0.5).astype(int)

    result = {
        'model': name,
        'accuracy': accuracy_score(labels, preds),
        'precision': precision_score(labels, preds),
        'recall': recall_score(labels, preds),
        'f1': f1_score(labels, preds),
        'roc_auc': roc_auc_score(labels, probs),
    }
    print(f"[{name}] test — " + ", ".join(f"{k}={v:.4f}" for k, v in result.items() if k != 'model'))
    print(f"[{name}] confusion matrix:\n{confusion_matrix(labels, preds)}")
    return result, history

## 5. Run both conditions

In [ ]:
result_no_aug, history_no_aug = train_and_evaluate('CNN_1D_no_augmentation', augment_train=False)

In [ ]:
result_aug, history_aug = train_and_evaluate('CNN_1D_with_augmentation', augment_train=True)

## 6. Side-by-side comparison

In [ ]:
comparison = pd.DataFrame([result_no_aug, result_aug])
comparison

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(history_no_aug['train_loss'], label='train'); ax[0].plot(history_no_aug['val_loss'], label='val')
ax[0].set_title('No augmentation'); ax[0].set_xlabel('epoch'); ax[0].legend()
ax[1].plot(history_aug['train_loss'], label='train'); ax[1].plot(history_aug['val_loss'], label='val')
ax[1].set_title('With augmentation'); ax[1].set_xlabel('epoch'); ax[1].legend()
plt.tight_layout()
fig_path = os.path.join(DRIVE_ROOT, 'augmentation_ablation_curves.png')
plt.savefig(fig_path, dpi=150)
plt.show()
print(f"Saved to {fig_path}")

## 7. Append to the running results log

In [ ]:
if os.path.exists(RESULTS_PATH):
    prior = pd.read_csv(RESULTS_PATH)
    combined = pd.concat([prior, comparison], ignore_index=True)
else:
    combined = comparison

combined.to_csv(RESULTS_PATH, index=False)
print(f"Saved to {RESULTS_PATH}")
combined